# Confidence Score Analysis

This notebook performs **bootstrap confidence analysis** on the trained waste classification model.

It estimates:

- Mean confidence score for each class
- 95% bootstrap confidence interval
- Number of correctly classified samples

The analysis is performed using the **RealWaste zero-shot evaluation dataset** and supports the methodology described in **Section 3.2.4 (Data Analysis Techniques)**.

In [1]:
import json
import os

import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader

import config
from dataset import (
    get_clean_transforms,
    get_heavy_transforms,
    make_realwaste_splits,
)

from models import build_model

In [2]:
BACKBONE = "resnet50"

# Other options:
# BACKBONE = "efficientnet_b0"
# BACKBONE = "mobilenet_v3_large"

In [3]:
def bootstrap_ci(values,
                 num_bootstrap=2000,
                 ci=0.95):

    values = np.array(values)

    if len(values) == 0:
        return None, None, None

    rng = np.random.default_rng(config.RANDOM_SEED)

    means = []

    n = len(values)

    for _ in range(num_bootstrap):

        sample = rng.choice(
            values,
            size=n,
            replace=True
        )

        means.append(sample.mean())

    means = np.sort(means)

    lower_idx = int((1-ci)/2 * num_bootstrap)
    upper_idx = int((1+ci)/2 * num_bootstrap)

    return (
        float(values.mean()),
        float(means[lower_idx]),
        float(means[upper_idx]),
    )

In [4]:
@torch.no_grad()
def collect_confidences(backbone_name):

    device = config.DEVICE

    checkpoint = os.path.join(
        config.CHECKPOINT_DIR,
        f"{backbone_name}_trashnet_best.pt"
    )

    if not os.path.exists(checkpoint):
        raise FileNotFoundError(
            f"Checkpoint not found:\n{checkpoint}"
        )

    _, eval_tf = get_clean_transforms()

    _, finetune_tf = get_heavy_transforms()

    zero_shot_ds, _, _, _ = make_realwaste_splits(
        eval_tf,
        finetune_tf
    )

    loader = DataLoader(
        zero_shot_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    model, _ = build_model(
        backbone_name,
        len(config.TRASHNET_CLASSES),
        pretrained=False
    )

    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=device
        )
    )

    model = model.to(device)
    model.eval()

    per_class_confidences = {
        c: [] for c in config.TRASHNET_CLASSES
    }

    for images, labels in loader:

        images = images.to(device)

        outputs = model(images)

        probs = F.softmax(outputs, dim=1)

        confidences, predictions = probs.max(dim=1)

        for pred, conf, true_label in zip(
                predictions.cpu().numpy(),
                confidences.cpu().numpy(),
                labels.numpy()):

            if pred == true_label:

                class_name = config.TRASHNET_CLASSES[true_label]

                per_class_confidences[class_name].append(
                    float(conf)
                )

    return per_class_confidences

In [5]:
def run_confidence_analysis(backbone):

    per_class = collect_confidences(backbone)

    results = {}

    print("="*60)
    print("BOOTSTRAP CONFIDENCE ANALYSIS")
    print("="*60)

    for class_name, scores in per_class.items():

        mean_conf, lower, upper = bootstrap_ci(scores)

        results[class_name] = {

            "n_correct_predictions": len(scores),

            "mean_confidence": mean_conf,

            "ci_95_lower": lower,

            "ci_95_upper": upper,
        }

        if mean_conf is None:

            print(f"{class_name:15s} : No correct predictions")

        else:

            print(
                f"{class_name:15s}"
                f" | n={len(scores):4d}"
                f" | Mean={mean_conf:.3f}"
                f" | CI=[{lower:.3f}, {upper:.3f}]"
            )

    save_path = os.path.join(
        config.RESULTS_DIR,
        f"{backbone}_confidence_analysis.json"
    )

    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)

    print("\nResults saved to")
    print(save_path)

    return results

In [6]:
results = run_confidence_analysis(BACKBONE)

BOOTSTRAP CONFIDENCE ANALYSIS
cardboard       | n= 228 | Mean=0.901 | CI=[0.880, 0.922]
glass           | n= 112 | Mean=0.889 | CI=[0.857, 0.917]
metal           | n= 385 | Mean=0.888 | CI=[0.873, 0.902]
paper           | n=  91 | Mean=0.848 | CI=[0.815, 0.879]
plastic         | n= 237 | Mean=0.819 | CI=[0.791, 0.843]
trash           | n=  30 | Mean=0.737 | CI=[0.668, 0.801]

Results saved to
e:\wasteClassificationProject_FIXED\wasteClassificationProject\outputs\resnet50_confidence_analysis.json


In [7]:
results

{'cardboard': {'n_correct_predictions': 228,
  'mean_confidence': 0.9011742002085635,
  'ci_95_lower': 0.8798017869131607,
  'ci_95_upper': 0.921711414921702},
 'glass': {'n_correct_predictions': 112,
  'mean_confidence': 0.8888992068490812,
  'ci_95_lower': 0.8574783381606851,
  'ci_95_upper': 0.9171396970216718},
 'metal': {'n_correct_predictions': 385,
  'mean_confidence': 0.8878534421518252,
  'ci_95_lower': 0.8732491987866241,
  'ci_95_upper': 0.9016665076280569},
 'paper': {'n_correct_predictions': 91,
  'mean_confidence': 0.8479645131053505,
  'ci_95_lower': 0.8147052513374077,
  'ci_95_upper': 0.8785369717157804},
 'plastic': {'n_correct_predictions': 237,
  'mean_confidence': 0.8185269167403129,
  'ci_95_lower': 0.791439420437511,
  'ci_95_upper': 0.8432532926149006},
 'trash': {'n_correct_predictions': 30,
  'mean_confidence': 0.736808305978775,
  'ci_95_lower': 0.667718290289243,
  'ci_95_upper': 0.8014039427042008}}

In [8]:
import pandas as pd

df = pd.DataFrame(results).T

df

,n_correct_predictions,mean_confidence,ci_95_lower,ci_95_upper
cardboard,228.0,0.901174,0.879802,0.921711
glass,112.0,0.888899,0.857478,0.917140
metal,385.0,0.887853,0.873249,0.901667
paper,91.0,0.847965,0.814705,0.878537
plastic,237.0,0.818527,0.791439,0.843253
trash,30.0,0.736808,0.667718,0.801404
